In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
import os
from joblib import Parallel, delayed

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
class ProceduralTerrainDataset(Dataset):
    def __init__(self, num_samples=5000, grid_size=64):
        self.num_samples = num_samples
        self.grid_size = grid_size
        
        self.num_cores = os.cpu_count()
        print(f"Generating {self.num_samples} terrains using {self.num_cores} CPU cores...")
        self.data = self._generate_batch(self.num_samples)
        print("Generation complete!")

    def _generate_terrain_slice(self):
        noise = np.random.rand(self.grid_size, self.grid_size)
        sigma = np.random.uniform(1.2, 2.2)
        smoothed = gaussian_filter(noise, sigma=sigma)
        normalized = (smoothed - smoothed.min()) / (smoothed.max() - smoothed.min())
        return normalized.astype(np.float32)[np.newaxis, :, :] # Add channel dim immediately

    def _generate_batch(self, count):
        batch = Parallel(n_jobs=self.num_cores)(
            delayed(self._generate_terrain_slice)() for _ in range(count)
        )
        return torch.tensor(np.array(batch))

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x = self.data[idx]
        return x, x

In [ ]:
class TerrainHD_VAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Flatten()
        )

        self.fc_mu = nn.Linear(64 * 8 * 8, latent_dim)
        self.fc_logvar = nn.Linear(64 * 8 * 8, latent_dim)
        self.fc_decode_input = nn.Linear(latent_dim, 64 * 8 * 8)

        self.decoder = nn.Sequential(
            nn.Unflatten(1, (64, 8, 8)),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1), nn.Sigmoid() 
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        features = self.encoder(x)
        mu = self.fc_mu(features)
        logvar = self.fc_logvar(features)
        z = self.reparameterize(mu, logvar)
        
        decoded_features = self.fc_decode_input(z)
        reconstruction = self.decoder(decoded_features)
        return reconstruction, mu, logvar

In [ ]:
def vae_loss_fn(recon_x, x, mu, logvar, kl_weight=0.01):
    recon_loss = nn.functional.binary_cross_entropy(recon_x, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + (kl_weight * kl_loss), recon_loss, kl_loss

In [ ]:
dataset = ProceduralTerrainDataset(num_samples=5000, grid_size=64)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
torch.set_num_threads(os.cpu_count())
print(f"PyTorch compute threads set to: {torch.get_num_threads()}")

In [ ]:
model = TerrainHD_VAE(latent_dim=32)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 50

print("\n--- Starting CPU Training ---")
for epoch in range(epochs):
    model.train()
    total_loss, total_recon, total_kl = 0, 0, 0

    for batch_idx, (data, _) in enumerate(dataloader):
        optimizer.zero_grad()
        
        recon_batch, mu, logvar = model(data)
        
        loss, recon_loss, kl_loss = vae_loss_fn(recon_batch, data, mu, logvar)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()

    n = len(dataloader.dataset)
    print(f"Epoch {epoch+1}/{epochs} | Total Loss: {total_loss/n:.2f} "
          f"-> (Recon: {total_recon/n:.2f}, KL: {total_kl/n:.2f})")
          
print("--- Training Complete ---")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import PillowWriter
import numpy as np
import torch

checkpoint_path = '/kaggle/working/terrain_hd_vae.pth'
torch.save(model.state_dict(), checkpoint_path)
print(f"Model weights saved successfully to {checkpoint_path}")

print("\n--- Rendering 3D Animated Latent Walk ---")
model.eval()

GRID_SIZE = 64 
x = np.linspace(0, 1, GRID_SIZE)
y = np.linspace(0, 1, GRID_SIZE)
X, Y = np.meshgrid(x, y)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.set_axis_off()

frames = 60
alphas = np.concatenate([np.linspace(0, 1, frames//2), np.linspace(1, 0, frames//2)])

with torch.no_grad():
    z1 = torch.randn(1, model.latent_dim)
    z2 = torch.randn(1, model.latent_dim)
    
    def update(frame_idx):
        ax.clear()
        ax.set_axis_off()
        ax.set_zlim(0, 1.5)
        
        alpha = alphas[frame_idx]
        z_interp = (1 - alpha) * z1 + alpha * z2
        
        decoded_features = model.fc_decode_input(z_interp)
        terrain = model.decoder(decoded_features).squeeze().numpy()
        
        surf = ax.plot_surface(X, Y, terrain, cmap='terrain', rstride=1, cstride=1, linewidth=0, antialiased=True)
        ax.set_title(f"Latent Morph: {(alpha*100):.0f}%", pad=-20)
        return surf,

    print("Compiling frames... (this takes about 30 seconds)")
    from matplotlib.animation import FuncAnimation
    ani = FuncAnimation(fig, update, frames=frames, blit=False)
    
    output_gif = '/kaggle/working/3D_terrain_walk.gif'
    ani.save(output_gif, dpi=100, writer=PillowWriter(fps=30))
    print(f"STUNNING SUCCESS: 3D Animation saved to {output_gif}")
    plt.close(fig)